In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/rearc/rearc_landing/raw/bls/"
    )
)

In [0]:
@dp.materialized_view(name="bronze_bls_data")
def bronze_bls_data():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .csv(
            "/Volumes/workspace/rearc/rearc_landing/raw/bls/pr.data.1.AllData"
        )
    )

In [0]:
@dp.materialized_view(name="bronze_bls_series")
def bronze_bls_series():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .csv(
            "/Volumes/workspace/rearc/rearc_landing/raw/bls/pr.series"
        )
    )

In [0]:
@dp.materialized_view(name="bronze_bls_sector")
def bronze_bls_sector():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .csv(
            "/Volumes/workspace/rearc/rearc_landing/raw/bls/pr.sector"
        )
    )

In [0]:
@dp.materialized_view(name="bronze_bls_measure")
def bronze_bls_measure():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .csv(
            "/Volumes/workspace/rearc/rearc_landing/raw/bls/pr.measure"
        )
    )

In [0]:
@dp.materialized_view(name="bronze_bls_period")
def bronze_bls_period():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .csv(
            "/Volumes/workspace/rearc/rearc_landing/raw/bls/pr.period"
        )
    )

In [0]:
@dp.materialized_view(name="bronze_bls_seasonal")
def bronze_bls_seasonal():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .csv(
            "/Volumes/workspace/rearc/rearc_landing/raw/bls/pr.seasonal"
        )
    )

In [0]:
@dp.materialized_view(name="bronze_bls_duration")
def bronze_bls_duration():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .csv(
            "/Volumes/workspace/rearc/rearc_landing/raw/bls/pr.duration"
        )
    )

In [0]:
@dp.materialized_view(name="bronze_datausa_population")
def bronze_datausa_population():
    return (
        spark.read
        .option("multiLine", "true")
        .json(
            "/Volumes/workspace/rearc/rearc_landing/raw/datausa/population.json"
        )
        .selectExpr("explode(data) AS record")
        .select("record.*")
    )

In [0]:
@dp.materialized_view(name="bronze_bls_class")
def bronze_bls_class():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .csv(
            "/Volumes/workspace/rearc/rearc_landing/raw/bls/pr.class"
        )
    )

In [0]:
@dp.materialized_view(name="bronze_bls_footnote")
def bronze_bls_footnote():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .csv(
            "/Volumes/workspace/rearc/rearc_landing/raw/bls/pr.footnote"
        )
    )

In [0]:
@dp.materialized_view(name="bronze_bls_contacts")
def bronze_bls_contacts():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .csv(
            "/Volumes/workspace/rearc/rearc_landing/raw/bls/pr.contacts"
        )
    )

In [0]:
@dp.materialized_view(name="silver_bls_data")
def silver_bls_data():
    return (
        dp.read("bronze_bls_data")
        .select(
            F.trim("series_id").alias("series_id"),
            F.col("year").cast("int").alias("year"),
            F.trim("period").alias("period"),
            F.col("value").cast("double").alias("value"),
            F.trim("footnote_codes").alias("footnote_codes")
        )
        .filter(
            F.col("series_id").isNotNull()
            & F.col("year").isNotNull()
            & F.col("value").isNotNull()
        )
    )

In [0]:
@dp.materialized_view(name="silver_datausa_population")
def silver_datausa_population():
    return (
        dp.read("bronze_datausa_population")
        .select(
            F.col("Year").cast("int").alias("year"),
            F.trim("Nation").alias("nation"),
            F.col("Population").cast("long").alias("population")
        )
        .filter(
            F.col("Year").isNotNull()
            & F.col("Population").isNotNull()
        )
    )

In [0]:
@dp.materialized_view(name="silver_bls_series")
def silver_bls_series():
    return (
        dp.read("bronze_bls_series")
        .select(
            F.trim("series_id").alias("series_id"),
            F.trim("seasonal").alias("seasonal"),
            F.trim("series_title").alias("series_title"),
            F.col("begin_year").cast("int").alias("begin_year"),
            F.trim("begin_period").alias("begin_period"),
            F.col("end_year").cast("int").alias("end_year"),
            F.trim("end_period").alias("end_period")
        )
        .filter(F.col("series_id").isNotNull())
        .dropDuplicates(["series_id"])
    )

In [0]:
@dp.materialized_view(name="gold_q1_population_stats")
def gold_q1_population_stats():
    population = dp.read("silver_datausa_population")

    filtered = population.filter(
        (F.col("nation") == "United States")
        & F.col("year").between(2013, 2018)
    )

    return filtered.agg(
        F.countDistinct("year").alias("total_years"),
        F.min("year").alias("start_year"),
        F.max("year").alias("end_year"),
        F.avg("population").alias("mean_population"),
        F.stddev_pop("population").alias("stddev_population")
    )

In [0]:
@dp.materialized_view(name="gold_q2_productivity")
def gold_q2_productivity():
    quarterly = (
        dp.read("silver_bls_data")
        .filter(
            F.col("period").rlike("^Q0[1-4]$")
        )
        .groupBy("series_id", "year")
        .agg(
            F.sum("value").alias("annual_value")
        )
    )

    from pyspark.sql.window import Window

    ranking = Window.partitionBy("series_id").orderBy(
        F.desc("annual_value"),
        F.asc("year")
    )

    ranked = (
        quarterly
        .withColumn("rank", F.row_number().over(ranking))
        .filter(F.col("rank") == 1)
    )

    return (
        ranked
        .join(
            dp.read("silver_bls_series").select(
                "series_id", "series_title"
            ),
            on="series_id",
            how="left"
        )
        .select(
            "series_id",
            "series_title",
            "year",
            "annual_value"
        )
    )

In [0]:
@dp.materialized_view(name="gold_q3_bls_population")
def gold_q3_bls_population():
    productivity = (
        dp.read("silver_bls_data")
        .filter(
            (F.trim("series_id") == "PRS30006032")
            & (F.col("period") == "Q01")
        )
        .select(
            "year",
            F.col("value").alias("annual_value")
        )
    )

    population = (
        dp.read("silver_datausa_population")
        .filter(F.col("nation") == "United States")
        .select("year", "population")
    )

    return (
        productivity
        .join(population, on="year", how="left")
        .select("year", "annual_value", "population")
    )

In [0]:
@dp.expect_or_fail(
    "valid_bls_series_id",
    "series_id IS NOT NULL AND TRIM(series_id) <> ''"
)
@dp.materialized_view(name="silver_bls_data_quality")
def silver_bls_data_quality():
    return (
        dp.read("silver_bls_data")
        .filter(
            F.col("year").between(1900, 2100)
            & F.col("value").isNotNull()
        )
    )

In [0]:
print("Lakeflow source definitions:")
print("Bronze: 11 datasets")
print("Silver: 3 datasets")
print("Gold: 3 datasets")
print("Quality: 1 dataset")